In [ ]:
import pandas as pd
import numpy as np
import yaml

from dotenv import load_dotenv

from ml_project.utils import get_project_directories
from ml_project.cleaning import clean_data
from ml_project.data import split_dataset
from ml_project.feature_engineering import (
    generate_features, handle_outliers, detect_outliers,
    impute_missing_data, one_hot_encoding, scaling
)
from ml_project.models import create_model, train_model
from ml_project.evaluation import evaluate_model

In [ ]:
directory_paths_dict = get_project_directories()

env_path = directory_paths_dict["root"] / ".env"
load_dotenv(dotenv_path=env_path)

cfg_path = directory_paths_dict["configs"] / "data.yaml"
with cfg_path.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

In [ ]:
locations_df = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["locations"])
sensors_metadata_df = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["sensors_metadata"])
sensors_measurements_df = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["sensors_measurements"])
weather_df = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["meteostat"]["weather_daily"])
dataframes_dict_raw = {
    "locations": locations_df,
    "sensors_metadata": sensors_metadata_df,
    "sensors_measurements": sensors_measurements_df,
    "weather": weather_df
}

for name, df in dataframes_dict_raw.items():
    print(f"Loaded {name}: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
cleaned_data_dict = clean_data(dataframes_dict_raw)
cleaned_df = cleaned_data_dict["cleaned"]

print(f"Cleaned dataframe shape: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")
print(f"Columns: {list(cleaned_df.columns)}")

In [ ]:
missing_counts = cleaned_df.isna().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print("Missing values per column:")
    for col, count in missing_cols.items():
        pct = (count / len(cleaned_df)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("No missing values found")

In [ ]:
outliers = detect_outliers(cleaned_df)

if outliers:
    print(f"Columns with outliers: {list(outliers.keys())}")
    total_outliers = sum(len(v) for v in outliers.values())
    print(f"Total outliers detected: {total_outliers}")
    for col, outlier_list in outliers.items():
        print(f"  {col}: {len(outlier_list)} outliers")

    cleaned_df = handle_outliers(cleaned_df, method="cap")
    print(f"Outliers capped. Shape after handling: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")
else:
    print("No outliers detected")

In [ ]:
cleaned_df = impute_missing_data(cleaned_df)
missing_after = cleaned_df.isna().sum().sum()

print(f"Missing values after imputation: {missing_after}")
print(f"Shape after imputation: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")

In [ ]:
featured_df = generate_features(cleaned_df)
new_features = set(featured_df.columns) - set(cleaned_df.columns)

print(f"Generated {len(new_features)} new features")
if new_features:
    print(f"New features: {list(new_features)}")
print(f"Shape after feature generation: {featured_df.shape[0]} rows, {featured_df.shape[1]} columns")

In [ ]:
encoded_df = one_hot_encoding(featured_df)
new_encoded_cols = encoded_df.shape[1] - featured_df.shape[1]

print(f"Added {new_encoded_cols} columns from one-hot encoding")
print(f"Shape after one-hot encoding: {encoded_df.shape[0]} rows, {encoded_df.shape[1]} columns")

In [ ]:
train_test_split_pct = 0.3 
x_train, x_test, y_train, y_test = split_dataset(encoded_df, test_size=train_test_split_pct)
print(f"Train set: {len(x_train) if x_train is not None else 0} rows, Test set: {len(x_test) if x_test is not None else 0} rows")

In [ ]:
x_train_scaled, train_scaler = scaling(x_train)
x_test_scaled, _ = scaling(x_test, existing_scaler=train_scaler)
print(f"Scaling complete. Train shape: {x_train_scaled.shape if x_train_scaled is not None else None}, Test shape: {x_test_scaled.shape if x_test_scaled is not None else None}")

In [ ]:
model_cfg_path = directory_paths_dict["configs"] / "model.yaml"
with model_cfg_path.open("r", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f) or {}

model_params = model_cfg.get("model_params", {})
print(f"Model parameters:")
print(yaml.dump(model_params, default_flow_style=False) if model_params else "  (empty - using defaults)")

In [ ]:
model = create_model(model_params)
print(f"Model created: {type(model).__name__ if model else 'None'}")

In [ ]:
trained_model = train_model(model, x_train)
print(f"Model training complete")

In [ ]:
metrics = evaluate_model(trained_model, (x_test, y_test))
print(f"Model evaluation complete: {metrics}")

In [ ]:
# Final dataframe preview
encoded_df.head()

In [ ]:
# If you want to save the file locally
to_save_df = encoded_df               # Change this

to_save_df.to_csv(directory_paths_dict["data_interim"] / "temp.csv", index=False)